Lien vers la version Colab : https://colab.research.google.com/drive/1zz5CwMYQCuFfj-n2QMqjikqAksKzN2PE?authuser=2#scrollTo=NavM1rF6AqGs

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight

!pip install -q xgboost
from xgboost import XGBClassifier

# Étape 1: Chargement des données


On charge les données extraites de la vidéo

Chaque ligne correspond à **une souris à un instant donné** et contient :
- `frame_id`, `instance_id` : identifiant de frame et de souris
- `x, y, w, h` : bounding box
- `x0..x7`, `y0..y7` : keypoints de chaque souris
- `following_score`, `escape_score`, `approach_score` : annotations de comportement


In [ ]:
df = pd.read_csv("/content/train_algo1_with_features.csv")
print("df shape :", df.shape)
display(df.head())
print()
print(df.info())

df shape : (1183052, 28)


,following_score,escape_score,approach_score,instance_id,frame_id,video_id,x0,y0,x5,y5,...,vy,speed,delta_dist,angle,delta_angle,fs,es,as,is_event,behavior_full
0,0,0,0,0,0,13-10-02,1106.0,987.0,1054.0,937.0,...,0.0,0.0,0.0,1.916916,0.0,0,0,0,0,none
1,0,0,0,1,0,13-10-02,1751.0,1200.0,1803.0,1136.0,...,0.0,0.0,0.0,-1.968295,0.0,0,0,0,0,none
2,0,0,0,2,0,13-10-02,1699.0,866.0,1635.0,946.0,...,0.0,0.0,0.0,1.173298,0.0,0,0,0,0,none
3,0,0,0,3,0,13-10-02,2247.0,582.0,2309.0,616.0,...,0.0,0.0,0.0,-3.031452,0.0,0,0,0,0,none
4,-1,-1,-1,4,0,13-10-02,-1.0,-1.0,-1.0,-1.0,...,0.0,0.0,0.0,0.000000,0.0,0,0,0,0,none



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1183052 entries, 0 to 1183051
Data columns (total 28 columns):
 #   Column           Non-Null Count    Dtype  
---  ------           --------------    -----  
 0   following_score  1183052 non-null  int64  
 1   escape_score     1183052 non-null  int64  
 2   approach_score   1183052 non-null  int64  
 3   instance_id      1183052 non-null  int64  
 4   frame_id         1183052 non-null  int64  
 5   video_id         1183052 non-null  object 
 6   x0               1183052 non-null  float64
 7   y0               1183052 non-null  float64
 8   x5               1183052 non-null  float64
 9   y5               1183052 non-null  float64
 10  x                1183052 non-null  float64
 11  y                1183052 non-null  float64
 12  w                1183052 non-null  float64
 13  h                1183052 non-null  float64
 14  rel_dx           1183052 non-null  float64
 15  rel_dy           1183052 non-null  float64
 16  dist_to_partner  

#Étape 2: Création des labels `behavior_full` et `is_event`


1. **Label multi-classe `behavior_full`** :
   - `approach` si `approach_score == 1`
   - `escape` si `escape_score == 1`
   - `follow` si `following_score == 1`
   - `none` sinon (aucun comportement annoté)

2. **Label binaire `is_event`** :
   - `0` si `behavior_full == "none"`
   - `1` sinon car il y a un comportement


In [ ]:
def get_behavior_full(row):
    if row["approach_score"] == 1:
        return "approach"
    if row["escape_score"] == 1:
        return "escape"
    if row["following_score"] == 1:
        return "follow"
    return "none"

df["behavior_full"] = df.apply(get_behavior_full, axis=1)
df["is_event"] = df["behavior_full"].apply(lambda v: 0 if v == "none" else 1)

print("Distribution behavior_full :")
print(df["behavior_full"].value_counts())
print("\nDistribution is_event (0=none, 1=event) :")
print(df["is_event"].value_counts())


Distribution behavior_full :
behavior_full
none        1163632
approach      18184
escape          707
follow          529
Name: count, dtype: int64

Distribution is_event (0=none, 1=event) :
is_event
0    1163632
1      19420
Name: count, dtype: int64


On constate un fort déséquilibre : beaucoup de `none`, peu de `escape` / `follow`.

#Étape 3: Création de features relationnel, vitesse, dynamique


On va rajouter des informations plus explicites pour le modèle :

1. **Relationnel (par frame)** :
   - pour chaque souris, trouver la souris la plus proche
   - calculer `rel_dx`, `rel_dy` : vecteur relatif
   - `dist_to_partner` : distance à cette souris

2. **Vitesse (par souris dans le temps)** :
   - `vx`, `vy` : variations de x et y d'une frame à l'autre
   - `speed` : norme de la vitesse

3. **Dynamique relationnelle** :
   - `angle` : angle vers la souris la plus proche
   - `delta_dist` : variation de distance vers la partenaire
   - `delta_angle` : variation de l'angle


In [ ]:
df_all = df.copy().sort_values(["frame_id", "instance_id"]).reset_index(drop=True)

#  Features relationnelles
partner_ids = np.full(len(df_all), -1, dtype=int)
rel_dx = np.zeros(len(df_all), dtype=float)
rel_dy = np.zeros(len(df_all), dtype=float)
dist_to_partner = np.zeros(len(df_all), dtype=float)

for frame_id, idx in df_all.groupby("frame_id").groups.items():
    idx = np.array(list(idx))
    if len(idx) < 2:
        dist_to_partner[idx] = np.nan
        rel_dx[idx] = np.nan
        rel_dy[idx] = np.nan
        partner_ids[idx] = -1
        continue

    xs = df_all.loc[idx, "x"].to_numpy()
    ys = df_all.loc[idx, "y"].to_numpy()
    inst_ids = df_all.loc[idx, "instance_id"].to_numpy()

    for k, row_idx in enumerate(idx):
        dx = xs - xs[k]
        dy = ys - ys[k]
        dist = np.sqrt(dx**2 + dy**2)
        dist[k] = np.inf

        nearest = dist.argmin()
        partner_ids[row_idx] = inst_ids[nearest]
        rel_dx[row_idx] = dx[nearest]
        rel_dy[row_idx] = dy[nearest]
        dist_to_partner[row_idx] = dist[nearest]

df_all["partner_instance_id"] = partner_ids
df_all["rel_dx"] = rel_dx
df_all["rel_dy"] = rel_dy
df_all["dist_to_partner"] = dist_to_partner

#  Vitesse
df_all = df_all.sort_values(["instance_id", "frame_id"]).reset_index(drop=True)
df_all["vx"] = df_all.groupby("instance_id")["x"].diff().fillna(0)
df_all["vy"] = df_all.groupby("instance_id")["y"].diff().fillna(0)
df_all["speed"] = np.sqrt(df_all["vx"]**2 + df_all["vy"]**2)

#  Dynamique relationnelle
df_all["angle"] = np.arctan2(df_all["rel_dy"], df_all["rel_dx"])
df_all["delta_dist"] = df_all.groupby("instance_id")["dist_to_partner"].diff().fillna(0)
df_all["delta_angle"] = df_all.groupby("instance_id")["angle"].diff().fillna(0)

# NaN -> 0 pour les frames sans partenaire
df_all[["rel_dx", "rel_dy", "dist_to_partner", "angle"]] = df_all[["rel_dx", "rel_dy", "dist_to_partner", "angle"]].fillna(0)

display(
    df_all[[
        "frame_id", "instance_id", "behavior_full",
        "partner_instance_id", "rel_dx", "rel_dy", "dist_to_partner",
        "vx", "vy", "speed",
        "delta_dist", "angle", "delta_angle"
    ]].head(15)
)


,frame_id,instance_id,behavior_full,partner_instance_id,rel_dx,rel_dy,dist_to_partner,vx,vy,speed,delta_dist,angle,delta_angle
0,0,0,none,5,-361.5,-7.5,361.577792,0.0,0.0,0.000000,0.000000,-3.120849,0.000000
1,0,0,none,22,144.5,-64.0,158.038761,1112.5,24.5,1112.769743,-203.539031,-0.416939,2.703909
2,0,0,none,3,145.5,-100.0,176.550984,-60.0,-287.5,293.694144,18.512223,-0.602141,-0.185202
3,1,0,none,5,-322.0,-13.5,322.282873,-1053.5,258.0,1084.631850,145.731889,-3.099692,-2.497550
4,1,0,none,22,134.0,-61.5,147.438970,1116.5,27.5,1116.838619,-174.843902,-0.430276,2.669416
5,1,0,none,3,154.5,-106.5,187.649940,-73.0,-283.0,292.263580,40.210970,-0.603522,-0.173246
6,2,0,none,5,-287.0,-16.0,287.445647,-1050.0,249.5,1079.235957,99.795707,-3.085901,-2.482379
7,2,0,none,22,121.5,-55.5,133.575821,1128.5,26.0,1128.799473,-153.869826,-0.428486,2.657415
8,2,0,none,3,157.0,-110.5,191.987630,-80.0,-273.5,284.960085,58.411809,-0.613286,-0.184800
9,3,0,none,5,-265.5,-25.0,266.674427,-1050.5,248.0,1079.376788,74.686797,-3.047708,-2.434421




- On a maintenant des informations relationnelles (position par rapport à la souris la plus proche).
- On a des informations de mouvement (vitesse) et de variations (delta distance, delta angle).
- Ces features sont importantes pour distinguer :
  - `approach` : distance qui diminue ;
  - `escape` : distance qui augmente ;
  - `follow` : distance et angle relativement stables.


#Étape 4: Définition de la liste de features

**Objectif de cette étape**

Construire une liste unique `full_features` contenant toutes les variables explicatives que l’on va donner aux modèles :

- bounding box : `x, y, w, h`
- keypoints : `x0..x7`, `y0..y7`
- relationnel : `rel_dx, rel_dy, dist_to_partner`
- vitesse : `vx, vy, speed`
- dynamique relationnelle : `delta_dist, angle, delta_angle`


In [ ]:
keypoints = []
for i in range(8):
    keypoints += [f"x{i}", f"y{i}"]

base_features = ["x", "y", "w", "h"]
geom_features = base_features + keypoints
relation_features = ["rel_dx", "rel_dy", "dist_to_partner"]
velocity_features = ["vx", "vy", "speed"]
dynamic_features = ["delta_dist", "angle", "delta_angle"]

full_features = geom_features + relation_features + velocity_features + dynamic_features

print("Nombre total de features :", len(full_features))
print(full_features)


Nombre total de features : 29
['x', 'y', 'w', 'h', 'x0', 'y0', 'x1', 'y1', 'x2', 'y2', 'x3', 'y3', 'x4', 'y4', 'x5', 'y5', 'x6', 'y6', 'x7', 'y7', 'rel_dx', 'rel_dy', 'dist_to_partner', 'vx', 'vy', 'speed', 'delta_dist', 'angle', 'delta_angle']


La liste `full_features` contient toutes les variables utilisées comme entrée.
Ces 29 features combinent géométrie, relationnel et dynamique, ce qui donne une représentation riche du comportement.


#Étape: 5 Modèle binaire : `none` vs `event`


On va entraîner un **RandomForest binaire** qui décide si à une frame donnée :

- il ne se passe rien (`none`),
- ou s'il y a un comportement (`event`).

Ce modèle doit :
- bien reconnaître les frames `none` (peu de faux positifs)
- détecter la majorité des frames `event`.

Ce modèle servira de filtre avant la classification fine.


In [ ]:
X_bin = df_all[full_features].copy()
y_bin = df_all["is_event"].copy()

X_bin_train, X_bin_test, y_bin_train, y_bin_test = train_test_split(
    X_bin, y_bin,
    test_size=0.20,
    stratify=y_bin,
    random_state=42
)

print("Répartition train (0=none,1=event) :", np.bincount(y_bin_train))
print("Répartition test  (0=none,1=event) :", np.bincount(y_bin_test))

rf_bin = RandomForestClassifier(
    n_estimators=600,
    max_depth=None,
    class_weight={0: 1.0, 1: 5.0},  # on pondère plus les events
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42
)

rf_bin.fit(X_bin_train, y_bin_train)
y_bin_pred = rf_bin.predict(X_bin_test)

print("\nRapport de classification :\n")
print(classification_report(y_bin_test, y_bin_pred, target_names=["none", "event"]))

print("\nMatrice de confusion :\n")
print(confusion_matrix(y_bin_test, y_bin_pred))


KeyError: "['x1', 'y1', 'x2', 'y2', 'x3', 'y3', 'x4', 'y4', 'x6', 'y6', 'x7', 'y7'] not in index"



- Le modèle  atteint une très forte précision sur la classe `none` (frames neutres).
- Le recall sur `event` est élevé 0.78, ce qui signifie qu'on récupère la majorité des comportements annotés.
- Ce modèle filtre les frames neutres avant la classification fine.


# Étape 6: Modèle multi-classes sur les événements (`approach`, `escape`, `follow`)


On entraîne un XGBoost uniquement sur les frames pour lesquelles `is_event = 1`.

- On retire complètement la classe `none` à cette étape.
- On prédit entre 3 classes : `approach`, `escape`, `follow`.

Le but est de maximiser la performance sur les classes rares `escape` et `follow`.

À la fin on obtient un modèle spécialisé dans la classification fine des comportements.


In [ ]:
# On ne garde que les frames où un comportement existe
df_events = df_all[df_all["is_event"] == 1].copy()

X_event = df_events[full_features].copy()
y_event = df_events["behavior_full"].copy()

le_event = LabelEncoder()
y_event_enc = le_event.fit_transform(y_event)

print("Classes :", le_event.classes_)

X_ev_train, X_ev_test, y_ev_train, y_ev_test = train_test_split(
    X_event, y_event_enc,
    test_size=0.20,
    stratify=y_event_enc,
    random_state=42
)

print("Train :", X_ev_train.shape, "Test :", X_ev_test.shape)

xgb_event = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softmax",
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42
)

xgb_event.fit(X_ev_train, y_ev_train)
y_ev_pred = xgb_event.predict(X_ev_test)

print("\nRapport de classification :\n")
print(classification_report(y_ev_test, y_ev_pred, target_names=le_event.classes_))

print("\nMatrice de confusion :\n")
print(confusion_matrix(y_ev_test, y_ev_pred))



- Sur les frames où un événement existe, le modèle atteint une précision très haute (environs 0.99).
- `approach` est reconnue presque parfaitement.
- Les classes rares `escape` et `follow` obtiennent des scores F1 élevé.
- Ce modèle est donc très adapté pour la classification fine des comportements sociaux.


#Étape 7: Pipeline de prédiction complet


On combine les deux modèles en un pipeline complet :

1. On applique d'abord le modèle random forest :
   - s'il prédit `none` alors on retourne directement `none`
   - s'il prédit `event` alors on passe à l'étape suivante

2. On applique le XGBoost :
   - et on prédit `approach`, `escape` ou `follow`.

À la fin on aura une fonction `predict_behavior_frame` qui prend un vecteur de features (1 frame) et renvoie une des 4 classes : `none`, `approach`, `escape`, `follow`.


In [ ]:
def predict_behavior_frame(feature_vector_1d):
    """
    feature_vector_1d : np.array shape (29,)
    Retourne une étiquette parmi : 'none', 'approach', 'escape', 'follow'
    """
    fv = feature_vector_1d.reshape(1, -1)

    # Étape 1 : prédiction binaire none/event
    pred_bin = rf_bin.predict(fv)[0]
    if pred_bin == 0:
        return "none"

    # Étape 2 : classification fine de l'événement
    pred_event = xgb_event.predict(fv)[0]
    label = le_event.inverse_transform([pred_event])[0]
    return label


In [ ]:
# ============================================================
# Évaluation globale du pipeline sur la validation
# ============================================================

from sklearn.metrics import classification_report, confusion_matrix

print("\n=== ÉVALUATION GLOBALE DU PIPELINE (none / approach / escape / follow) ===")

# 1) Vraies étiquettes sur le bloc de validation
y_true_full = df_sorted.loc[val_mask, "behavior_full"].to_numpy()

# 2) Features correspondantes
X_val_full = df_sorted.loc[val_mask, feature_cols].to_numpy()

# 3) Prédictions pipeline (RF binaire + XGBoost)
y_pred_full = []
for i in range(X_val_full.shape[0]):
    y_pred_full.append(predict_behavior_frame(X_val_full[i, :]))
y_pred_full = np.array(y_pred_full)

# 4) Ordre des classes pour avoir une matrice lisible
labels_order = ["none", "approach", "escape", "follow"]

print("\n--- Classification Report GLOBAL ---")
print(classification_report(
    y_true_full,
    y_pred_full,
    labels=labels_order,
    target_names=labels_order
))

cm_global = confusion_matrix(y_true_full, y_pred_full, labels=labels_order)
print("\n--- Matrice de confusion globale ---")
print("Ordre des classes :", labels_order)
print(cm_global)



- La fonction `predict_behavior_frame` encapsule le pipeline complet.
- Sur quelques exemples on vérifie que les prédictions sont cohérentes avec le label `behavior_full`.


# Étape 8:  Évaluation globale du pipeline sur un échantillon


On évalue les performances du pipeline sur un sous-échantillon de frames (environs 10 000) pour :

- mesurer la qualité globale sur `none`, `approach`, `escape`, `follow`


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# ==================
# 1) SPLIT TEMPOREL
# ==================

# 1) On trie par frame_id (important)
df_all_sorted = df_all.sort_values("frame_id").reset_index(drop=True)

# 2) On découpe la vidéo en 80% début (train) / 20% fin (val)
unique_frames = df_all_sorted["frame_id"].unique()
n_frames = len(unique_frames)
cut_idx = int(0.8 * n_frames)

train_frames = unique_frames[:cut_idx]
val_frames   = unique_frames[cut_idx:]

print(f"Nombre total de frames : {n_frames}")
print(f"Frames train : {train_frames[0]} → {train_frames[-1]}")
print(f"Frames val   : {val_frames[0]} → {val_frames[-1]}")

# 3) Masques temporels
mask_train_time = df_all_sorted["frame_id"].isin(train_frames)
mask_val_time   = df_all_sorted["frame_id"].isin(val_frames)

print("Nb lignes train (toutes classes) :", mask_train_time.sum())
print("Nb lignes val   (toutes classes) :", mask_val_time.sum())


# ============================
# 2) MODÈLE BINAIRE RF (none vs event)
# ============================

X_bin_all = df_all_sorted[full_features].to_numpy()
y_bin_all = df_all_sorted["is_event"].to_numpy()

X_train_bin = X_bin_all[mask_train_time.to_numpy()]
y_train_bin = y_bin_all[mask_train_time.to_numpy()]

X_val_bin = X_bin_all[mask_val_time.to_numpy()]
y_val_bin = y_bin_all[mask_val_time.to_numpy()]

print("RF binaire - X_train :", X_train_bin.shape, "X_val :", X_val_bin.shape)

from sklearn.ensemble import RandomForestClassifier

rf_bin = RandomForestClassifier(
    n_estimators=600,
    max_depth=None,
    class_weight={0: 1.0, 1: 5.0},
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42
)

rf_bin.fit(X_train_bin, y_train_bin)

print("\n=== Performance RF binaire sur bloc temporel de validation ===")
y_val_bin_pred = rf_bin.predict(X_val_bin)
print(classification_report(y_val_bin, y_val_bin_pred, target_names=["none", "event"]))
print(confusion_matrix(y_val_bin, y_val_bin_pred))


# ============================
# 3) MODÈLE XGBoost (approach / escape / follow)
# ============================

mask_event = (df_all_sorted["is_event"] == 1)

mask_train_event = mask_train_time & mask_event
mask_val_event   = mask_val_time & mask_event

X_event_all = df_all_sorted[full_features].to_numpy()
y_event_str = df_all_sorted["behavior_full"].to_numpy()  # 'approach','escape','follow','none'

# On enlève les 'none'
X_train_event = X_event_all[mask_train_event.to_numpy()]
y_train_event = y_event_str[mask_train_event.to_numpy()]

X_val_event = X_event_all[mask_val_event.to_numpy()]
y_val_event = y_event_str[mask_val_event.to_numpy()]

print("\nEvents train :", X_train_event.shape[0], "Events val :", X_val_event.shape[0])

# Encodage labels pour XGBoost
from sklearn.preprocessing import LabelEncoder
le_event = LabelEncoder()
y_train_event_enc = le_event.fit_transform(y_train_event)
y_val_event_enc   = le_event.transform(y_val_event)

print("Classes (ordre) :", le_event.classes_)

import xgboost as xgb

# Gestion du déséquilibre
from collections import Counter
class_counts = Counter(y_train_event_enc)
total = sum(class_counts.values())
class_weight = {cls: total / (len(class_counts) * cnt) for cls, cnt in class_counts.items()}
print("Poids de classes :", class_weight)

# On convertit en scale_pos_weight au niveau des samples via sample_weight
sample_weight = np.array([class_weight[c] for c in y_train_event_enc])

xgb_event = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softmax",
    num_class=len(le_event.classes_),
    tree_method="hist",
    eval_metric="mlogloss",
    random_state=42,
)

xgb_event.fit(X_train_event, y_train_event_enc, sample_weight=sample_weight)

print("\n=== Performance XGBoost (event only) sur bloc temporel de validation ===")
y_val_pred_enc = xgb_event.predict(X_val_event)
print(classification_report(y_val_event_enc, y_val_pred_enc, target_names=le_event.classes_))
print(confusion_matrix(y_val_event_enc, y_val_pred_enc))
